In [ ]:
# In[1]:

import pandas as pd

# Load CSVs (files are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes as required (keep original integer timestamps too)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to compute file-level metadata
def file_metadata(df, kpi_col=None, message_check=False):
    rows = len(df)
    min_ts = int(df['timestamp'].min()) if rows > 0 else None
    max_ts = int(df['timestamp'].max()) if rows > 0 else None
    cmdb_ids = sorted(df['cmdb_id'].dropna().unique().tolist()) if 'cmdb_id' in df.columns else []
    kpi_names = sorted(df[kpi_col].dropna().unique().tolist()) if (kpi_col and kpi_col in df.columns) else None
    message_info = None
    if message_check:
        has_message = 'message' in df.columns
        non_empty_count = 0
        if has_message:
            # count non-empty messages robustly
            non_empty_count = int(df['message'].dropna().astype(str).str.strip().replace({'nan':''}).apply(lambda x: x != '').sum())
        message_info = {'has_message_column': has_message, 'non_empty_messages': non_empty_count}
    return {'rows': rows, 'min_timestamp': min_ts, 'max_timestamp': max_ts, 'cmdb_ids': cmdb_ids, 'kpi_names': kpi_names, 'message_info': message_info}

# Compute metadata for each file
metric_meta = file_metadata(metric_df, kpi_col='kpi_name')
trace_meta = file_metadata(trace_df, kpi_col='trace_name')
log_meta = file_metadata(log_df, kpi_col='log_name')
error_meta = file_metadata(error_df, message_check=True)

# Grouped counts per (cmdb_id, KPI) with earliest/latest timestamp, top 20 by count
metric_groups = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(count=('timestamp', 'size'),
         ts_min=('timestamp', 'min'),
         ts_max=('timestamp', 'max'))
    .reset_index()
    .sort_values('count', ascending=False)
)
metric_top20 = metric_groups.head(20)

trace_groups = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(count=('timestamp', 'size'),
         ts_min=('timestamp', 'min'),
         ts_max=('timestamp', 'max'))
    .reset_index()
    .sort_values('count', ascending=False)
)
trace_top20 = trace_groups.head(20)

log_groups = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(count=('timestamp', 'size'),
         ts_min=('timestamp', 'min'),
         ts_max=('timestamp', 'max'))
    .reset_index()
    .sort_values('count', ascending=False)
)
log_top20 = log_groups.head(20)

# Compose a compact summary structure
summary = {
    'metric.csv': {
        'metadata': {k:v for k,v in metric_meta.items() if k != 'message_info'},
        'top_groups_count_preview': metric_top20[['cmdb_id','kpi_name','count','ts_min','ts_max']]
    },
    'trace.csv': {
        'metadata': {k:v for k,v in trace_meta.items() if k != 'message_info'},
        'top_groups_count_preview': trace_top20[['cmdb_id','trace_name','count','ts_min','ts_max']]
    },
    'log.csv': {
        'metadata': {k:v for k,v in log_meta.items() if k != 'message_info'},
        'top_groups_count_preview': log_top20[['cmdb_id','log_name','count','ts_min','ts_max']]
    },
    'error_logs.csv': {
        'metadata': {k:v for k,v in error_meta.items() if k != 'kpi_names'},
        'message_info': error_meta['message_info']
    }
}

# Display compact results (as variables, per instructions)
summary, metric_top20[['cmdb_id','kpi_name','count','ts_min','ts_max']], trace_top20[['cmdb_id','trace_name','count','ts_min','ts_max']], log_top20[['cmdb_id','log_name','count','ts_min','ts_max']]

```
Out[1]:
```
```python
# Summarize the previously computed metadata and top-group previews in plain English.
# Reuse variables from earlier execution: metric_meta, trace_meta, log_meta, error_meta,
# metric_top20, trace_top20, log_top20.

def format_top(groups_df, name_col, topn=5):
    rows = []
    for _, r in groups_df.head(topn).iterrows():
        rows.append(f"{r['cmdb_id']} / {r[name_col]} (count={int(r['count'])}, ts_min={int(r['ts_min'])}, ts_max={int(r['ts_max'])})")
    return rows

metric_cmdb_count = len(metric_meta['cmdb_ids'])
trace_cmdb_count = len(trace_meta['cmdb_ids'])
log_cmdb_count = len(log_meta['cmdb_ids'])
error_has_message = error_meta['message_info']['has_message_column']
error_non_empty = error_meta['message_info']['non_empty_messages']

summary = (
    "Summary of telemetry files:\n\n"
    "metric.csv:\n"
    f"- rows: {metric_meta['rows']}, min_timestamp: {metric_meta['min_timestamp']}, max_timestamp: {metric_meta['max_timestamp']}\n"
    f"- unique cmdb_id count: {metric_cmdb_count} (examples: {', '.join(metric_meta['cmdb_ids'][:6])}...)\n"
    f"- unique kpi_names: {metric_meta['kpi_names']}\n"
    "- top (cmdb_id / kpi) groups by row count (top 5):\n"
    + "\n".join("  - " + s for s in format_top(metric_top20, 'kpi_name', 5))
    + "\n\n"
    "trace.csv:\n"
    f"- rows: {trace_meta['rows']}, min_timestamp: {trace_meta['min_timestamp']}, max_timestamp: {trace_meta['max_timestamp']}\n"
    f"- unique cmdb_id count: {trace_cmdb_count} (examples: {', '.join(trace_meta['cmdb_ids'][:6])}...)\n"
    f"- unique trace_names count: {len(trace_meta['kpi_names'])}\n"
    "- top (cmdb_id / trace_name) groups by row count (top 5):\n"
    + "\n".join("  - " + s for s in format_top(trace_top20, 'trace_name', 5))
    + "\n\n"
    "log.csv:\n"
    f"- rows: {log_meta['rows']}, min_timestamp: {log_meta['min_timestamp']}, max_timestamp: {log_meta['max_timestamp']}\n"
    f"- unique cmdb_id count: {log_cmdb_count} (examples: {', '.join(log_meta['cmdb_ids'][:6])}...)\n"
    f"- unique log_names: {log_meta['kpi_names']}\n"
    "- top (cmdb_id / log_name) groups by row count (top 5):\n"
    + "\n".join("  - " + s for s in format_top(log_top20, 'log_name', 5))
    + "\n\n"
    "error_logs.csv:\n"
    f"- rows: {error_meta['rows']}, min_timestamp: {error_meta['min_timestamp']}, max_timestamp: {error_meta['max_timestamp']}\n"
    f"- has 'message' column: {error_has_message}, non-empty messages: {error_non_empty}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'metric.csv': {'metadata': {'rows': 9053, 'min_timestamp': 1706022840, 'max_timestamp': 1706024280, 'cmdb_ids': ['ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-mysql', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-news-service', 'ts-notification-mongo', 'ts-notification-service', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-payment-service', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-rebook-service', 'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticket-office-service', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo', 'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo', 'ts-user-service', 'ts-verification-code-service', 'ts-voucher-mysql', 'ts-voucher-service'], 'kpi_names': ['cpu', 'diskio', 'error', 'latency-50', 'latency-90', 'mem', 'socket', 'workload']}, 'top_groups_count_preview':                          cmdb_id    kpi_name  count      ts_min      ts_max
366           ts-voucher-service      socket     25  1706022840  1706024280
0    ts-admin-basic-info-service         cpu     25  1706022840  1706024280
1    ts-admin-basic-info-service      diskio     25  1706022840  1706024280
2    ts-admin-basic-info-service  latency-50     25  1706022840  1706024280
3    ts-admin-basic-info-service  latency-90     25  1706022840  1706024280
4    ts-admin-basic-info-service         mem     25  1706022840  1706024280
5    ts-admin-basic-info-service      socket     25  1706022840  1706024280
6    ts-admin-basic-info-service    workload     25  1706022840  1706024280
7         ts-admin-order-service         cpu     25  1706022840  1706024280
8         ts-admin-order-service      diskio     25  1706022840  1706024280
9         ts-admin-order-service         mem     25  1706022840  1706024280
10        ts-admin-order-service      socket     25  1706022840  1706024280
11        ts-admin-route-service         cpu     25  1706022840  1706024280
12        ts-admin-route-service      diskio     25  1706022840  1706024280
13        ts-admin-route-service         mem     25  1706022840  1706024280
327             ts-travel2-mongo         mem     25  1706022840  1706024280
328             ts-travel2-mongo      socket     25  1706022840  1706024280
329           ts-travel2-service         cpu     25  1706022840  1706024280
330           ts-travel2-service      diskio     25  1706022840  1706024280
331           ts-travel2-service  latency-50     25  1706022840  1706024280}, 'trace.csv': {'metadata': {'rows': 18760, 'min_timestamp': 1706022840, 'max_timestamp': 1706024280, 'cmdb_ids': ['root', 'ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-service', 'ts-auth-service', 'ts-basic-service', 'ts-config-service', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-service', 'ts-food-map-service', 'ts-food-service', 'ts-inside-payment-service', 'ts-order-other-service', 'ts-order-service', 'ts-payment-service', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-service', 'ts-route-service', 'ts-seat-service', 'ts-security-service', 'ts-station-service', 'ts-ticketinfo-service', 'ts-train-service', 'ts-travel-service', 'ts-travel2-service', 'ts-user-service'], 'kpi_names': ['trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.from_ts-admin-basic-info-service.duration_mean', 'trace.from_ts-admin-basic-info-service.duration_p95', 'trace.from_ts-admin-basic-info-service.error_rate', 'trace.from_ts-admin-basic-info-service.row_count', 'trace.from_ts-admin-travel-service.duration_mean', 'trace.from_ts-admin-travel-service.duration_p95', 'trace.from_ts-admin-travel-service.error_rate', 'trace.from_ts-admin-travel-service.row_count', 'trace.from_ts-assurance-service.duration_mean', 'trace.from_ts-assurance-service.duration_p95', 'trace.from_ts-assurance-service.error_rate', 'trace.from_ts-assurance-service.row_count', 'trace.from_ts-auth-service.duration_mean', 'trace.from_ts-auth-service.duration_p95', 'trace.from_ts-auth-service.error_rate', 'trace.from_ts-auth-service.row_count', 'trace.from_ts-basic-service.duration_mean', 'trace.from_ts-basic-service.duration_p95', 'trace.from_ts-basic-service.error_rate', 'trace.from_ts-basic-service.row_count', 'trace.from_ts-config-service.duration_mean', 'trace.from_ts-config-service.duration_p95', 'trace.from_ts-config-service.error_rate', 'trace.from_ts-config-service.row_count', 'trace.from_ts-consign-price-service.duration_mean', 'trace.from_ts-consign-price-service.duration_p95', 'trace.from_ts-consign-price-service.error_rate', 'trace.from_ts-consign-price-service.row_count', 'trace.from_ts-consign-service.duration_mean', 'trace.from_ts-consign-service.duration_p95', 'trace.from_ts-consign-service.error_rate', 'trace.from_ts-consign-service.row_count', 'trace.from_ts-contacts-service.duration_mean', 'trace.from_ts-contacts-service.duration_p95', 'trace.from_ts-contacts-service.error_rate', 'trace.from_ts-contacts-service.row_count', 'trace.from_ts-food-map-service.duration_mean', 'trace.from_ts-food-map-service.duration_p95', 'trace.from_ts-food-map-service.error_rate', 'trace.from_ts-food-map-service.row_count', 'trace.from_ts-food-service.duration_mean', 'trace.from_ts-food-service.duration_p95', 'trace.from_ts-food-service.error_rate', 'trace.from_ts-food-service.row_count', 'trace.from_ts-inside-payment-service.duration_mean', 'trace.from_ts-inside-payment-service.duration_p95', 'trace.from_ts-inside-payment-service.error_rate', 'trace.from_ts-inside-payment-service.row_count', 'trace.from_ts-order-other-service.duration_mean', 'trace.from_ts-order-other-service.duration_p95', 'trace.from_ts-order-other-service.error_rate', 'trace.from_ts-order-other-service.row_count', 'trace.from_ts-order-service.duration_mean', 'trace.from_ts-order-service.duration_p95', 'trace.from_ts-order-service.error_rate', 'trace.from_ts-order-service.row_count', 'trace.from_ts-payment-service.duration_mean', 'trace.from_ts-payment-service.duration_p95', 'trace.from_ts-payment-service.error_rate', 'trace.from_ts-payment-service.row_count', 'trace.from_ts-preserve-other-service.duration_mean', 'trace.from_ts-preserve-other-service.duration_p95', 'trace.from_ts-preserve-other-service.error_rate', 'trace.from_ts-preserve-other-service.row_count', 'trace.from_ts-preserve-service.duration_mean', 'trace.from_ts-preserve-service.duration_p95', 'trace.from_ts-preserve-service.error_rate', 'trace.from_ts-preserve-service.row_count', 'trace.from_ts-price-service.duration_mean', 'trace.from_ts-price-service.duration_p95', 'trace.from_ts-price-service.error_rate', 'trace.from_ts-price-service.row_count', 'trace.from_ts-route-service.duration_mean', 'trace.from_ts-route-service.duration_p95', 'trace.from_ts-route-service.error_rate', 'trace.from_ts-route-service.row_count', 'trace.from_ts-seat-service.duration_mean', 'trace.from_ts-seat-service.duration_p95', 'trace.from_ts-seat-service.error_rate', 'trace.from_ts-seat-service.row_count', 'trace.from_ts-security-service.duration_mean', 'trace.from_ts-security-service.duration_p95', 'trace.from_ts-security-service.error_rate', 'trace.from_ts-security-service.row_count', 'trace.from_ts-station-service.duration_mean', 'trace.from_ts-station-service.duration_p95', 'trace.from_ts-station-service.error_rate', 'trace.from_ts-station-service.row_count', 'trace.from_ts-ticketinfo-service.duration_mean', 'trace.from_ts-ticketinfo-service.duration_p95', 'trace.from_ts-ticketinfo-service.error_rate', 'trace.from_ts-ticketinfo-service.row_count', 'trace.from_ts-train-service.duration_mean', 'trace.from_ts-train-service.duration_p95', 'trace.from_ts-train-service.error_rate', 'trace.from_ts-train-service.row_count', 'trace.from_ts-travel-service.duration_mean', 'trace.from_ts-travel-service.duration_p95', 'trace.from_ts-travel-service.error_rate', 'trace.from_ts-travel-service.row_count', 'trace.from_ts-travel2-service.duration_mean', 'trace.from_ts-travel2-service.duration_p95', 'trace.from_ts-travel2-service.error_rate', 'trace.from_ts-travel2-service.row_count', 'trace.from_ts-user-service.duration_mean', 'trace.from_ts-user-service.duration_p95', 'trace.from_ts-user-service.error_rate', 'trace.from_ts-user-service.row_count', 'trace.to_ts-admin-basic-info-service.duration_mean', 'trace.to_ts-admin-basic-info-service.duration_p95', 'trace.to_ts-admin-basic-info-service.error_rate', 'trace.to_ts-admin-basic-info-service.row_count', 'trace.to_ts-admin-travel-service.duration_mean', 'trace.to_ts-admin-travel-service.duration_p95', 'trace.to_ts-admin-travel-service.error_rate', 'trace.to_ts-admin-travel-service.row_count', 'trace.to_ts-assurance-service.duration_mean', 'trace.to_ts-assurance-service.duration_p95', 'trace.to_ts-assurance-service.error_rate', 'trace.to_ts-assurance-service.row_count', 'trace.to_ts-auth-service.duration_mean', 'trace.to_ts-auth-service.duration_p95', 'trace.to_ts-auth-service.error_rate', 'trace.to_ts-auth-service.row_count', 'trace.to_ts-basic-service.duration_mean', 'trace.to_ts-basic-service.duration_p95', 'trace.to_ts-basic-service.error_rate', 'trace.to_ts-basic-service.row_count', 'trace.to_ts-config-service.duration_mean', 'trace.to_ts-config-service.duration_p95', 'trace.to_ts-config-service.error_rate', 'trace.to_ts-config-service.row_count', 'trace.to_ts-consign-price-service.duration_mean', 'trace.to_ts-consign-price-service.duration_p95', 'trace.to_ts-consign-price-service.error_rate', 'trace.to_ts-consign-price-service.row_count', 'trace.to_ts-consign-service.duration_mean', 'trace.to_ts-consign-service.duration_p95', 'trace.to_ts-consign-service.error_rate', 'trace.to_ts-consign-service.row_count', 'trace.to_ts-contacts-service.duration_mean', 'trace.to_ts-contacts-service.duration_p95', 'trace.to_ts-contacts-service.error_rate', 'trace.to_ts-contacts-service.row_count', 'trace.to_ts-food-map-service.duration_mean', 'trace.to_ts-food-map-service.duration_p95', 'trace.to_ts-food-map-service.error_rate', 'trace.to_ts-food-map-service.row_count', 'trace.to_ts-food-service.duration_mean', 'trace.to_ts-food-service.duration_p95', 'trace.to_ts-food-service.error_rate', 'trace.to_ts-food-service.row_count', 'trace.to_ts-inside-payment-service.duration_mean', 'trace.to_ts-inside-payment-service.duration_p95', 'trace.to_ts-inside-payment-service.error_rate', 'trace.to_ts-inside-payment-service.row_count', 'trace.to_ts-order-other-service.duration_mean', 'trace.to_ts-order-other-service.duration_p95', 'trace.to_ts-order-other-service.error_rate', 'trace.to_ts-order-other-service.row_count', 'trace.to_ts-order-service.duration_mean', 'trace.to_ts-order-service.duration_p95', 'trace.to_ts-order-service.error_rate', 'trace.to_ts-order-service.row_count', 'trace.to_ts-payment-service.duration_mean', 'trace.to_ts-payment-service.duration_p95', 'trace.to_ts-payment-service.error_rate', 'trace.to_ts-payment-service.row_count', 'trace.to_ts-preserve-other-service.duration_mean', 'trace.to_ts-preserve-other-service.duration_p95', 'trace.to_ts-preserve-other-service.error_rate', 'trace.to_ts-preserve-other-service.row_count', 'trace.to_ts-preserve-service.duration_mean', 'trace.to_ts-preserve-service.duration_p95', 'trace.to_ts-preserve-service.error_rate', 'trace.to_ts-preserve-service.row_count', 'trace.to_ts-price-service.duration_mean', 'trace.to_ts-price-service.duration_p95', 'trace.to_ts-price-service.error_rate', 'trace.to_ts-price-service.row_count', 'trace.to_ts-route-service.duration_mean', 'trace.to_ts-route-service.duration_p95', 'trace.to_ts-route-service.error_rate', 'trace.to_ts-route-service.row_count', 'trace.to_ts-seat-service.duration_mean', 'trace.to_ts-seat-service.duration_p95', 'trace.to_ts-seat-service.error_rate', 'trace.to_ts-seat-service.row_count', 'trace.to_ts-security-service.duration_mean', 'trace.to_ts-security-service.duration_p95', 'trace.to_ts-security-service.error_rate', 'trace.to_ts-security-service.row_count', 'trace.to_ts-station-service.duration_mean', 'trace.to_ts-station-service.duration_p95', 'trace.to_ts-station-service.error_rate', 'trace.to_ts-station-service.row_count', 'trace.to_ts-ticketinfo-service.duration_mean', 'trace.to_ts-ticketinfo-service.duration_p95', 'trace.to_ts-ticketinfo-service.error_rate', 'trace.to_ts-ticketinfo-service.row_count', 'trace.to_ts-train-service.duration_mean', 'trace.to_ts-train-service.duration_p95', 'trace.to_ts-train-service.error_rate', 'trace.to_ts-train-service.row_count', 'trace.to_ts-travel-service.duration_mean', 'trace.to_ts-travel-service.duration_p95', 'trace.to_ts-travel-service.error_rate', 'trace.to_ts-travel-service.row_count', 'trace.to_ts-travel2-service.duration_mean', 'trace.to_ts-travel2-service.duration_p95', 'trace.to_ts-travel2-service.error_rate', 'trace.to_ts-travel2-service.row_count', 'trace.to_ts-user-service.duration_mean', 'trace.to_ts-user-service.duration_p95', 'trace.to_ts-user-service.error_rate', 'trace.to_ts-user-service.row_count']}, 'top_groups_count_preview':                 cmdb_id                                        trace_name  count      ts_min      ts_max
728  ts-travel2-service     trace.to_ts-order-other-service.duration_mean     25  1706022840  1706024280
729  ts-travel2-service      trace.to_ts-order-other-service.duration_p95     25  1706022840  1706024280
730  ts-travel2-service        trace.to_ts-order-other-service.error_rate     25  1706022840  1706024280
731  ts-travel2-service         trace.to_ts-order-other-service.row_count     25  1706022840  1706024280
732  ts-travel2-service           trace.to_ts-route-service.duration_mean     25  1706022840  1706024280
733  ts-travel2-service            trace.to_ts-route-service.duration_p95     25  1706022840  1706024280
734  ts-travel2-service              trace.to_ts-route-service.error_rate     25  1706022840  1706024280
735  ts-travel2-service               trace.to_ts-route-service.row_count     25  1706022840  1706024280
720  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25  1706022840  1706024280
721  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25  1706022840  1706024280
722  ts-travel2-service             trace.from_ts-seat-service.error_rate     25  1706022840  1706024280
723  ts-travel2-service              trace.from_ts-seat-service.row_count     25  1706022840  1706024280
724  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25  1706022840  1706024280
725  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25  1706022840  1706024280
726  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25  1706022840  1706024280
727  ts-travel2-service           trace.from_ts-travel2-service.row_count     25  1706022840  1706024280
712  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25  1706022840  1706024280
713  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25  1706022840  1706024280
714  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25  1706022840  1706024280
715  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25  1706022840  1706024280}, 'log.csv': {'metadata': {'rows': 2204, 'min_timestamp': 1706022840, 'max_timestamp': 1706024280, 'cmdb_ids': ['ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-basic-service', 'ts-config-mongo', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-notification-mongo', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-route-mongo', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-travel-mongo', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo'], 'kpi_names': ['log.error_count', 'log.row_count']}, 'top_groups_count_preview':                         cmdb_id         log_name  count      ts_min      ts_max
0   ts-admin-basic-info-service  log.error_count     25  1706022840  1706024280
1   ts-admin-basic-info-service    log.row_count     25  1706022840  1706024280
2       ts-admin-travel-service  log.error_count     25  1706022840  1706024280
3       ts-admin-travel-service    log.row_count     25  1706022840  1706024280
7          ts-assurance-service    log.row_count     25  1706022840  1706024280
6          ts-assurance-service  log.error_count     25  1706022840  1706024280
12             ts-basic-service  log.error_count     25  1706022840  1706024280
13             ts-basic-service    log.row_count     25  1706022840  1706024280
11              ts-auth-service    log.row_count     25  1706022840  1706024280
10              ts-auth-service  log.error_count     25  1706022840  1706024280
25          ts-contacts-service    log.row_count     25  1706022840  1706024280
24          ts-contacts-service  log.error_count     25  1706022840  1706024280
21           ts-consign-service    log.row_count     25  1706022840  1706024280
20           ts-consign-service  log.error_count     25  1706022840  1706024280
29          ts-food-map-service    log.row_count     25  1706022840  1706024280
28          ts-food-map-service  log.error_count     25  1706022840  1706024280
67              ts-seat-service    log.row_count     25  1706022840  1706024280
66              ts-seat-service  log.error_count     25  1706022840  1706024280
71          ts-security-service    log.row_count     25  1706022840  1706024280
70          ts-security-service  log.error_count     25  1706022840  1706024280}, 'error_logs.csv': {'metadata': {'rows': 0, 'min_timestamp': None, 'max_timestamp': None, 'cmdb_ids': [], 'message_info': {'has_message_column': True, 'non_empty_messages': 0}}, 'message_info': {'has_message_column': True, 'non_empty_messages': 0}}},                          cmdb_id    kpi_name  count      ts_min      ts_max
366           ts-voucher-service      socket     25  1706022840  1706024280
0    ts-admin-basic-info-service         cpu     25  1706022840  1706024280
1    ts-admin-basic-info-service      diskio     25  1706022840  1706024280
2    ts-admin-basic-info-service  latency-50     25  1706022840  1706024280
3    ts-admin-basic-info-service  latency-90     25  1706022840  1706024280
4    ts-admin-basic-info-service         mem     25  1706022840  1706024280
5    ts-admin-basic-info-service      socket     25  1706022840  1706024280
6    ts-admin-basic-info-service    workload     25  1706022840  1706024280
7         ts-admin-order-service         cpu     25  1706022840  1706024280
8         ts-admin-order-service      diskio     25  1706022840  1706024280
9         ts-admin-order-service         mem     25  1706022840  1706024280
10        ts-admin-order-service      socket     25  1706022840  1706024280
11        ts-admin-route-service         cpu     25  1706022840  1706024280
12        ts-admin-route-service      diskio     25  1706022840  1706024280
13        ts-admin-route-service         mem     25  1706022840  1706024280
327             ts-travel2-mongo         mem     25  1706022840  1706024280
328             ts-travel2-mongo      socket     25  1706022840  1706024280
329           ts-travel2-service         cpu     25  1706022840  1706024280
330           ts-travel2-service      diskio     25  1706022840  1706024280
331           ts-travel2-service  latency-50     25  1706022840  1706024280,                 cmdb_id                                        trace_name  count      ts_min      ts_max
728  ts-travel2-service     trace.to_ts-order-other-service.duration_mean     25  1706022840  1706024280
729  ts-travel2-service      trace.to_ts-order-other-service.duration_p95     25  1706022840  1706024280
730  ts-travel2-service        trace.to_ts-order-other-service.error_rate     25  1706022840  1706024280
731  ts-travel2-service         trace.to_ts-order-other-service.row_count     25  1706022840  1706024280
732  ts-travel2-service           trace.to_ts-route-service.duration_mean     25  1706022840  1706024280
733  ts-travel2-service            trace.to_ts-route-service.duration_p95     25  1706022840  1706024280
734  ts-travel2-service              trace.to_ts-route-service.error_rate     25  1706022840  1706024280
735  ts-travel2-service               trace.to_ts-route-service.row_count     25  1706022840  1706024280
720  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25  1706022840  1706024280
721  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25  1706022840  1706024280
722  ts-travel2-service             trace.from_ts-seat-service.error_rate     25  1706022840  1706024280
723  ts-travel2-service              trace.from_ts-seat-service.row_count     25  1706022840  1706024280
724  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25  1706022840  1706024280
725  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25  1706022840  1706024280
726  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25  1706022840  1706024280
727  ts-travel2-service           trace.from_ts-travel2-service.row_count     25  1706022840  1706024280
712  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25  1706022840  1706024280
713  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25  1706022840  1706024280
714  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25  1706022840  1706024280
715  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25  1706022840  1706024280,                         cmdb_id         log_name  count      ts_min      ts_max
0   ts-admin-basic-info-service  log.error_count     25  1706022840  1706024280
1   ts-admin-basic-info-service    log.row_count     25  1706022840  1706024280
2       ts-admin-travel-service  log.error_count     25  1706022840  1706024280
3       ts-admin-travel-service    log.row_count     25  1706022840  1706024280
7          ts-assurance-service    log.row_count     25  1706022840  1706024280
6          ts-assurance-service  log.error_count     25  1706022840  1706024280
12             ts-basic-service  log.error_count     25  1706022840  1706024280
13             ts-basic-service    log.row_count     25  1706022840  1706024280
11              ts-auth-service    log.row_count     25  1706022840  1706024280
10              ts-auth-service  log.error_count     25  1706022840  1706024280
25          ts-contacts-service    log.row_count     25  1706022840  1706024280
24          ts-contacts-service  log.error_count     25  1706022840  1706024280
21           ts-consign-service    log.row_count     25  1706022840  1706024280
20           ts-consign-service  log.error_count     25  1706022840  1706024280
29          ts-food-map-service    log.row_count     25  1706022840  1706024280
28          ts-food-map-service  log.error_count     25  1706022840  1706024280
67              ts-seat-service    log.row_count     25  1706022840  1706024280
66              ts-seat-service  log.error_count     25  1706022840  1706024280
71          ts-security-service    log.row_count     25  1706022840  1706024280
70          ts-security-service  log.error_count     25  1706022840  1706024280)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if available (assumed loaded previously): metric_df, trace_df, log_df
# Ensure timestamp_utc exists
for df in (metric_df, trace_df, log_df):
    if 'timestamp_utc' not in df.columns:
        df['timestamp_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Define candidate services
candidate_services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 2) Compute global P95 for metric.csv for candidate services (by cmdb_id, kpi_name) using full series
metric_candidate_full = metric_df[metric_df['cmdb_id'].isin(candidate_services)]
p95_metric = (
    metric_candidate_full
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 3) Compute global P95 for trace.csv and log.csv similarly
trace_candidate_full = trace_df[trace_df['cmdb_id'].isin(candidate_services)]
p95_trace = (
    trace_candidate_full
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

log_candidate_full = log_df[log_df['cmdb_id'].isin(candidate_services)]
p95_log = (
    log_candidate_full
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 4) Convert incident window bounds to UTC timestamps and filter rows to that window and candidate services
start_ts = pd.Timestamp("2024-01-23 15:11:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-23 15:41:51", tz='UTC')

metric_window = metric_df[
    (metric_df['cmdb_id'].isin(candidate_services)) &
    (metric_df['timestamp_utc'] >= start_ts) &
    (metric_df['timestamp_utc'] <= end_ts)
].copy()

trace_window = trace_df[
    (trace_df['cmdb_id'].isin(candidate_services)) &
    (trace_df['timestamp_utc'] >= start_ts) &
    (trace_df['timestamp_utc'] <= end_ts)
].copy()

log_window = log_df[
    (log_df['cmdb_id'].isin(candidate_services)) &
    (log_df['timestamp_utc'] >= start_ts) &
    (log_df['timestamp_utc'] <= end_ts)
].copy()

# Helper to compute breaches table for a windowed df given p95 table and key names
def compute_breaches(window_df, p95_df, key_cols, kpi_col_name):
    # incident counts per (cmdb_id, kpi)
    incident_counts = (
        window_df
        .groupby(key_cols, dropna=False)
        .size()
        .reset_index(name='incident_count')
    )
    # attach p95 per row by merging
    merged = window_df.merge(p95_df, on=key_cols, how='left')
    # boolean breach where value >= p95 (only when p95 is present)
    merged['is_breach'] = False
    merged.loc[merged['p95'].notna(), 'is_breach'] = merged.loc[merged['p95'].notna(), 'value'] >= merged.loc[merged['p95'].notna(), 'p95']
    # breaches subset
    breaches = merged[merged['is_breach']].copy()
    if breaches.empty:
        # return empty dataframe with expected columns
        cols = list(key_cols) + [kpi_col_name, 'p95', 'incident_count', 'breach_count', 'earliest_breach_ts', 'max_breach_value', 'max_breach_ts']
        return pd.DataFrame(columns=cols)
    # breach aggregates
    breach_agg = (
        breaches
        .groupby(key_cols, dropna=False)
        .agg(
            breach_count=('value','size'),
            earliest_breach_ts=('timestamp','min'),
            max_breach_value=('value','max')
        )
        .reset_index()
    )
    # get max_breach_ts via idxmax
    idx = breaches.groupby(key_cols)['value'].idxmax()
    max_ts_series = breaches.loc[idx, key_cols + ['timestamp']].set_index(key_cols)['timestamp']
    max_ts_series = max_ts_series.reset_index().rename(columns={'timestamp':'max_breach_ts'})
    # merge breach_agg with max_ts_series
    breach_agg = breach_agg.merge(max_ts_series, on=key_cols, how='left')
    # merge with incident_counts and p95 values
    # p95_df may have KPI name column; ensure it's present when merging
    merged_meta = breach_agg.merge(incident_counts, on=key_cols, how='left').merge(p95_df, on=key_cols, how='left')
    # rename KPI col
    if kpi_col_name not in merged_meta.columns:
        merged_meta = merged_meta.rename(columns={key_cols[-1]: kpi_col_name})
    # convert timestamps to integers
    merged_meta['earliest_breach_ts'] = merged_meta['earliest_breach_ts'].astype('Int64')
    merged_meta['max_breach_ts'] = merged_meta['max_breach_ts'].astype('Int64')
    # select and order columns
    result = merged_meta[[key_cols[0], key_cols[1], 'p95', 'incident_count', 'breach_count', 'earliest_breach_ts', 'max_breach_value', 'max_breach_ts']]
    result = result.rename(columns={key_cols[1]: kpi_col_name})
    # only consider breaches where breach_count > 0 (already true here), sort by breach_count desc then earliest_breach_ts asc
    result = result.sort_values(['breach_count','earliest_breach_ts'], ascending=[False, True]).reset_index(drop=True)
    return result

# 5) Compute breach tables
metric_breaches_df = compute_breaches(
    metric_window,
    p95_metric,
    key_cols=['cmdb_id','kpi_name'],
    kpi_col_name='kpi_name'
)

trace_breaches_df = compute_breaches(
    trace_window,
    p95_trace,
    key_cols=['cmdb_id','trace_name'],
    kpi_col_name='trace_name'
)

log_breaches_df = compute_breaches(
    log_window,
    p95_log,
    key_cols=['cmdb_id','log_name'],
    kpi_col_name='log_name'
)

# 6) Limit to top 20 rows per table (already sorted in compute_breaches)
metric_breaches_df = metric_breaches_df.head(20)
trace_breaches_df = trace_breaches_df.head(20)
log_breaches_df = log_breaches_df.head(20)

# Display compact tables as DataFrames (JSON-serializable when converted)
metric_breaches_df, trace_breaches_df, log_breaches_df

```
Out[2]:
```
```python
# Summarize the breach tables computed earlier in plain English.
# Reuse metric_breaches_df, trace_breaches_df, log_breaches_df from previous execution.

candidate_services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

def fmt_row(row, kpi_col):
    return (f"{row['cmdb_id']} / {row[kpi_col]}: p95={row['p95']}, "
            f"incident_count={int(row['incident_count'])}, breach_count={int(row['breach_count'])}, "
            f"earliest_breach_ts={int(row['earliest_breach_ts']) if pd.notna(row['earliest_breach_ts']) else None}, "
            f"max_breach_value={row['max_breach_value']}, max_breach_ts={int(row['max_breach_ts']) if pd.notna(row['max_breach_ts']) else None}")

# Top examples for each table (up to 3 each for concise summary)
metric_examples = [fmt_row(r, 'kpi_name') for _, r in metric_breaches_df.head(3).iterrows()]
trace_examples = [fmt_row(r, 'trace_name') for _, r in trace_breaches_df.head(3).iterrows()]
log_examples = [fmt_row(r, 'log_name') for _, r in log_breaches_df.head(3).iterrows()]

# Detect broad pattern: p95 equal zero causing many trivial "breaches"
def count_zero_p95(df, kpi_col):
    if df.empty:
        return 0
    return int((df['p95'] == 0).sum())

metric_zero_p95_count = count_zero_p95(metric_breaches_df, 'kpi_name')
trace_zero_p95_count = count_zero_p95(trace_breaches_df, 'trace_name')
log_zero_p95_count = count_zero_p95(log_breaches_df, 'log_name')

summary = (
    f"Candidate services: {candidate_services}\n\n"
    "Metric breaches (top examples):\n"
    + ("\n".join(" - " + s for s in metric_examples) if metric_examples else " - (no metric breaches)") + "\n\n"
    "Trace breaches (top examples):\n"
    + ("\n".join(" - " + s for s in trace_examples) if trace_examples else " - (no trace breaches)") + "\n\n"
    "Log breaches (top examples):\n"
    + ("\n".join(" - " + s for s in log_examples) if log_examples else " - (no log breaches)") + "\n\n"
    "Observations:\n"
    " - Many of the 'error' / 'error_rate' / 'log.error_count' KPIs have p95 = 0.0. Because the breach test used value >= p95,"
    " these KPIs show breach_count equal to the full incident_count (e.g., 25), which is a side-effect of p95 being zero rather than a strong anomaly signal.\n"
    f"   (metric zero-p95 count: {metric_zero_p95_count}, trace zero-p95 count: {trace_zero_p95_count}, log zero-p95 count: {log_zero_p95_count}).\n"
    " - True high-value breaches worth immediate investigation are the resource spikes (diskio, cpu, socket, workload, latency-90) found across services."
    " Examples from the metric table include:\n"
    "   * ts-auth-service / diskio spike to ~163612 at ts=1706022900\n"
    "   * ts-route-service / diskio spike to ~71108 at ts=1706022840\n"
    "   * ts-train-service / diskio spike to ~81419 at ts=1706022840\n"
    "   * CPU spikes for ts-order, ts-train, ts-route (small number of breaches each) around timestamps 1706022840-1706022960.\n\n"
    "Recommendation:\n"
    " - Treat the p95==0 error/error_rate/log.error_count findings as artifacts of the thresholding rule (value >= p95)."
    " Focus on the resource KPI breaches (diskio, cpu, socket, workload, latency-90) listed above and investigate corresponding hosts/containers and I/O/CPU activity at the provided timestamps.\n"
)

# Display the summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name            p95  incident_count  breach_count  earliest_breach_ts  max_breach_value  max_breach_ts
0    ts-order-service       error       0.000000              25            25          1706022840          0.000000     1706022840
1   ts-travel-service       error       0.000000               5             5          1706024040          0.000000     1706024040
2     ts-auth-service      diskio  105589.138477              25             2          1706022840     163612.389002     1706022900
3    ts-order-service         cpu       1.871464              25             2          1706022840          1.942378     1706022960
4    ts-route-service      diskio   64074.464308              25             2          1706022840      71108.024825     1706022840
5    ts-train-service         cpu       2.781999              25             2          1706022840          2.904730     1706023260
6    ts-train-service      diskio   62130.788191              25             2          1706022840      81419.271480     1706022840
7    ts-train-service      socket      16.950000              25             2          1706022840         17.100000     1706023440
8   ts-travel-service      diskio   62669.684162              25             2          1706022840      65132.668993     1706022840
9   ts-travel-service  latency-90       0.238015              25             2          1706022840          0.257156     1706022840
10  ts-travel-service      socket      17.036667              25             2          1706022840         17.777778     1706022840
11   ts-route-service      socket      13.933333              25             2          1706022900         14.000000     1706024100
12   ts-order-service      diskio   65683.352602              25             2          1706022960      66180.030602     1706022960
13   ts-order-service    workload       7.673707              25             2          1706022960          7.942183     1706022960
14   ts-route-service         cpu       4.579391              25             2          1706022960          4.623355     1706022960
15   ts-route-service    workload      36.459350              25             2          1706022960         36.634350     1706023260
16   ts-train-service    workload      22.828950              25             2          1706022960         23.302183     1706022960
17  ts-travel-service    workload      10.802470              25             2          1706022960         12.011033     1706022960
18    ts-auth-service    workload       2.883040              25             2          1706023080          3.028933     1706023080
19   ts-order-service      socket      12.846667              25             2          1706023080         13.116667     1706024100,               cmdb_id                                     trace_name  p95  incident_count  breach_count  earliest_breach_ts  max_breach_value  max_breach_ts
0     ts-auth-service                     trace.from_root.error_rate  0.0              25            25          1706022840               0.0     1706022840
1     ts-auth-service          trace.from_ts-auth-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
2     ts-auth-service            trace.to_ts-auth-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
3    ts-order-service         trace.from_ts-order-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
4    ts-order-service          trace.from_ts-seat-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
5    ts-order-service      trace.from_ts-security-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
6    ts-order-service        trace.from_ts-travel-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
7    ts-order-service           trace.to_ts-order-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
8    ts-route-service         trace.from_ts-basic-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
9    ts-route-service         trace.from_ts-route-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
10   ts-route-service        trace.from_ts-travel-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
11   ts-route-service       trace.from_ts-travel2-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
12   ts-route-service           trace.to_ts-route-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
13   ts-train-service         trace.from_ts-basic-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
14   ts-train-service         trace.from_ts-train-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
15   ts-train-service        trace.from_ts-travel-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
16   ts-train-service       trace.from_ts-travel2-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
17   ts-train-service           trace.to_ts-train-service.error_rate  0.0              25            25          1706022840               0.0     1706022840
18  ts-travel-service                     trace.from_root.error_rate  0.0              25            25          1706022840               0.0     1706022840
19  ts-travel-service  trace.from_ts-admin-travel-service.error_rate  0.0              25            25          1706022840               0.0     1706022840,              cmdb_id         log_name     p95  incident_count  breach_count  earliest_breach_ts  max_breach_value  max_breach_ts
0    ts-auth-service  log.error_count     0.0              25            25          1706022840                 0     1706022840
1   ts-order-service  log.error_count     0.0              25            25          1706022840                 0     1706022840
2   ts-route-service  log.error_count     0.0              25            25          1706022840                 0     1706022840
3  ts-travel-service  log.error_count     0.0              25            25          1706022840                 0     1706022840
4    ts-auth-service    log.row_count   343.6              25             2          1706023080               352     1706023080
5   ts-route-service    log.row_count  1091.0              25             2          1706023200              1097     1706023320
6   ts-order-service    log.row_count   440.8              25             2          1706023440               478     1706023680
7  ts-travel-service    log.row_count  1943.2              25             2          1706023440              2029     1706023680)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing variables: metric_df, metric_breaches_df
# Ensure timestamp_utc is present
if 'timestamp_utc' not in metric_df.columns:
    metric_df['timestamp_utc'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Candidate services
candidate_services = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 2) From metric_breaches_df keep only candidate services and p95 > 0
mb = metric_breaches_df.copy()  # reuse
mb_filtered = mb[mb['cmdb_id'].isin(candidate_services) & (mb['p95'] > 0)].reset_index(drop=True)

# Incident window (redefine to be safe)
start_ts = pd.Timestamp("2024-01-23 15:11:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-23 15:41:51", tz='UTC')

fault_rows = []

# 3) For each (cmdb_id, kpi_name) compute anomalous consecutive-minute faults
for _, row in mb_filtered.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = float(row['p95'])
    # a) Filter metric rows for this cmdb_id and kpi_name in incident window
    sel = metric_df[
        (metric_df['cmdb_id'] == cmdb) &
        (metric_df['kpi_name'] == kpi) &
        (metric_df['timestamp_utc'] >= start_ts) &
        (metric_df['timestamp_utc'] <= end_ts)
    ].copy()
    if sel.empty:
        continue
    # ensure integer timestamp column
    sel['ts_int'] = sel['timestamp'].astype(int)
    sel = sel.sort_values('ts_int')
    # b) Mark anomalous rows where value >= p95
    sel['is_anom'] = sel['value'] >= p95
    anoms = sel[sel['is_anom']].copy()
    if anoms.empty:
        continue
    # c) Group anomalies into consecutive-minute faults (diff == 60)
    anoms = anoms.sort_values('ts_int').reset_index(drop=True)
    anoms['ts_diff'] = anoms['ts_int'].diff().fillna(9999).astype(int)
    # new group where diff != 60
    anoms['group_id'] = (anoms['ts_diff'] != 60).cumsum()
    for gid, g in anoms.groupby('group_id'):
        start_ts_int = int(g['ts_int'].min())
        end_ts_int = int(g['ts_int'].max())
        count_points = int(g.shape[0])
        duration_minutes = int((end_ts_int - start_ts_int)//60 + 1)
        # max value and its timestamp (first occurrence of max)
        max_idx = g['value'].idxmax()
        max_value = float(g.loc[max_idx, 'value'])
        max_value_ts = int(g.loc[max_idx, 'ts_int'])
        severity_ratio = max_value / p95 if p95 != 0 else np.inf
        fault_rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'p95': p95,
            'start_ts': start_ts_int,
            'end_ts': end_ts_int,
            'duration_minutes': duration_minutes,
            'count_points': count_points,
            'max_value': max_value,
            'max_value_ts': max_value_ts,
            'severity_ratio': severity_ratio
        })

# 4) Filter out fault groups with severity_ratio <= 1.5
faults_df = pd.DataFrame(fault_rows)
if not faults_df.empty:
    faults_df = faults_df[faults_df['severity_ratio'] > 1.5].copy()
    # 5) Sort by severity_ratio desc and limit to top 20
    faults_df = faults_df.sort_values('severity_ratio', ascending=False).reset_index(drop=True).head(20)
    # Ensure integer timestamps are plain ints
    for c in ['start_ts','end_ts','max_value_ts','duration_minutes','count_points']:
        if c in faults_df.columns:
            faults_df[c] = faults_df[c].astype(int)
    faults_df['p95'] = faults_df['p95'].astype(float)
    faults_df['max_value'] = faults_df['max_value'].astype(float)
    faults_df['severity_ratio'] = faults_df['severity_ratio'].astype(float)
else:
    # empty frame with required columns
    faults_df = pd.DataFrame(columns=['cmdb_id','kpi_name','p95','start_ts','end_ts','duration_minutes','count_points','max_value','max_value_ts','severity_ratio'])

# Display compact fault table (JSON-serializable)
faults_df[['cmdb_id','kpi_name','p95','start_ts','end_ts','duration_minutes','count_points','max_value','max_value_ts','severity_ratio']].head(20)

```
Out[3]:
```
Summary of results (plain English):

- Candidate services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.
- After excluding KPI rows with p95 == 0 and grouping anomalous minutes into consecutive-minute faults, only one significant fault group passed the severity filter (severity_ratio > 1.5):

  1) Service: ts-auth-service  
     KPI: diskio  
     p95 (global): 105,589.138477  
     Fault window: 1706022840 → 1706022900 (UTC timestamps) = 2024-01-23 15:14:00 UTC → 2024-01-23 15:15:00 UTC  
     duration: 2 minutes, count_points: 2  
     max_value: 163,612.389002 at 1706022900 (2024-01-23 15:15:00 UTC)  
     severity_ratio = max_value / p95 ≈ 1.55

- Interpretation: ts-auth-service experienced a short but notable disk I/O spike (~1.55× the global P95) during the incident window (15:14–15:15 UTC). Other candidate-service KPI "breaches" were either p95==0 artifacts (commonly for error/error_rate/log.error_count) or produced weaker spikes (severity_ratio ≤ 1.5) and were filtered out. 

- Recommendation: Investigate ts-auth-service around 2024-01-23 15:14–15:15 UTC for disk I/O issues (logs, host/container I/O metrics, storage subsystem) since this is the strongest resource anomaly found.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name            p95    start_ts      end_ts  duration_minutes  count_points      max_value  max_value_ts  severity_ratio
0  ts-auth-service   diskio  105589.138477  1706022840  1706022900                 2             2  163612.389002    1706022900        1.549519```
```